<a href="https://colab.research.google.com/github/ManyaEleti/Evaluating-the-Impact-of-Feature-Engineering-on-Machine-Learning-Model-Performance/blob/main/01_SFT_LoRA_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.3 MB/s eta 0:00:00


In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

GPU available: True
GPU name: Tesla T4
GPU memory: 15.6 GB


In [4]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.6 MB/s eta 0:00:00


In [5]:
import importlib
import peft
importlib.reload(peft)
from peft import LoraConfig, get_peft_model, TaskType
print("PEFT loaded successfully")

PEFT loaded successfully


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

# Load model
model_name = "facebook/opt-125m"
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

# Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA applied!")

# Load dataset
print("Loading dataset...")
dataset = load_dataset("tatsu-lab/alpaca", split="train[:300]")

def format_prompt(example):
    if example["input"]:
        return f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

dataset = dataset.map(lambda x: {"text": format_prompt(x)})

# Train
print("Starting training...")
training_config = SFTConfig(
    output_dir="./sft_lora_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    dataset_text_field="text",
    max_seq_length=256,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_config,
    processing_class=tokenizer,
)

trainer.train()
print("Training complete!")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

trainable params: 294,912 || all params: 125,534,208 || trainable%: 0.2349
LoRA applied!
Loading dataset...


README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Starting training...


TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'. Did you mean 'max_length'?

In [7]:
# Train — fixed SFTConfig
print("Starting training...")
training_config = SFTConfig(
    output_dir="./sft_lora_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    dataset_text_field="text",
    max_length=256,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_config,
    processing_class=tokenizer,
)

trainer.train()
print("Training complete!")

Starting training...


Adding EOS to train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,3.035172
20,2.755699
30,2.637831


Training complete!


In [8]:
# Save the fine-tuned LoRA adapter
model.save_pretrained("./sft_lora_output/final_model")
tokenizer.save_pretrained("./sft_lora_output/final_model")
print("Model saved!")

# Test the fine-tuned model
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0
)

# Test with an instruction
prompt = "### Instruction:\nExplain what machine learning is in simple terms.\n\n### Response:\n"

output = generator(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

print("=" * 50)
print("FINE-TUNED MODEL OUTPUT:")
print("=" * 50)
print(output[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model saved!


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in OPTDecoderLayer. Setting `past_key_values=None`.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


FINE-TUNED MODEL OUTPUT:
### Instruction:
Explain what machine learning is in simple terms.

### Response:




In [9]:
# Better test with greedy decoding for small models
input_text = "### Instruction:\nWhat is the capital of France?\n\n### Response:\n"

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        temperature=1.0,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=" * 50)
print("FINE-TUNED MODEL OUTPUT:")
print("=" * 50)
print(response)
print("=" * 50)
print("\nNote: Small 125M model trained on 300 examples shows")
print("expected behavior — full results improve with more data/epochs")

FINE-TUNED MODEL OUTPUT:
### Instruction:
What is the capital of France?

### Response:
The The The the The " " ' ' ' '' or or or or or or or or or or or or or or or or or or or or or or or or or,- inininininininin///

Note: Small 125M model trained on 300 examples shows
expected behavior — full results improve with more data/epochs
